# AI Training Studio — Google Colab

เปิดลิงก์นี้บนบัญชี Colab ของตัวเอง แล้วเลือก **Runtime → Run all**
แต่ละคนได้สำเนาโปรแกรมและดิสก์ของตัวเอง ไม่ปนกับคนอื่น

ก่อนรัน: **Runtime → Change runtime type → GPU** (T4 หรือ L4)

เมื่อมีเวอร์ชันใหม่บน GitHub ให้ **Runtime → Restart session** แล้ว Run all อีกครั้ง โค้ดจะถูกแทนที่ด้วยเวอร์ชันล่าสุด ข้อมูลใน `/content/ai_training_workspace` ยังอยู่จนกว่า runtime จะถูกตัด

พอลิงก์โปรแกรมพร้อม โน้ตบุ๊กจะเปิดหน้าโปรแกรมให้เอง เซลล์ยังค้างสถานะ Executing ไว้ เพราะนั่นคือโปรแกรมที่กำลังทำงาน

ดิสก์ Colab เป็นพื้นที่ชั่วคราว ถ้า runtime ถูกยุติ ชุดข้อมูลและ checkpoint หาย ดาวน์โหลด `best.pt` จากหน้า Training หรือ Models ก่อนปิด

In [ ]:
# 1. ตรวจสอบ GPU และสภาพแวดล้อมของ Google Colab
import os, platform, subprocess, sys

print('Python:', platform.python_version())
res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print('⚠️ ไม่พบ GPU — แนะนำไปที่ Runtime -> Change runtime type -> เลือก T4 / L4 GPU')

In [ ]:
# 2. ดึงโค้ดล่าสุดจาก GitHub มาไว้บน runtime นี้
import os
import shutil
from pathlib import Path

REPO_URL = 'https://github.com/Atiruj2911/AI_Colab.git'
PROJECT_DIR = Path('/content/AI_Colab')
WORKSPACE_DIR = Path('/content/ai_training_workspace')

if (PROJECT_DIR / '.git').exists():
    !git -C "{PROJECT_DIR}" fetch origin
    !git -C "{PROJECT_DIR}" reset --hard origin/main
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    !git clone {REPO_URL} {PROJECT_DIR}

assert (PROJECT_DIR / 'app.py').exists(), f'ไม่พบ app.py ที่ {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
os.environ['AI_STUDIO_WORKSPACE'] = str(WORKSPACE_DIR)
print('Project Dir  :', PROJECT_DIR)
print('Workspace Dir:', WORKSPACE_DIR)

In [ ]:
# 3. ติดตั้ง Dependencies ที่กำหนดเวอร์ชันเข้ากันได้ (ไม่ทับ PyTorch CUDA ของ Colab)
!pip install -q -r requirements.txt

# พิมพ์เวอร์ชันแพ็กเกจทั้งหมดก่อนเปิดแอป
from pathlib import Path
from infrastructure.monitoring.system_monitor import format_startup_banner
print(format_startup_banner(Path(os.environ['AI_STUDIO_WORKSPACE'])))

In [ ]:
# 4. เปิด AI Training Studio แล้วเด้งหน้าโปรแกรมเมื่อลิงก์พร้อม
# เซลล์นี้จะค้าง Executing ไว้ ตลอดที่โปรแกรมทำงาน — อย่ากดหยุดเซลล์
import re
import subprocess
import sys

import threading

from IPython.display import HTML, display

URL_RE = re.compile(r"https://[0-9a-zA-Z-]+\.gradio\.live")


def pop_app(url: str) -> None:
    display(HTML(
        '<div style="margin:12px 0;padding:14px 16px;border:1px solid #2563EB;'
        'border-radius:8px;background:#EFF6FF;font-family:sans-serif">'
        '<div style="font-size:15px;font-weight:700;color:#1E293B">โปรแกรมพร้อมแล้ว</div>'
        f'<div style="margin-top:8px"><a href="{url}" target="_blank" '
        'style="font-size:16px;font-weight:700;color:#2563EB">เปิดหน้าโปรแกรม</a></div>'
        '<div style="margin-top:6px;color:#64748B;font-size:13px">'
        'ถ้าแท็บไม่เด้งเอง ให้กดลิงก์ด้านบน แล้วเปิดแท็บนี้ค้างไว้</div></div>'
    ))
    try:
        from google.colab import output
        output.eval_js(f"window.open({url!r}, '_blank'); 'opened'")
    except Exception as exc:
        print("เปิดแท็บอัตโนมัติไม่ได้:", exc)


proc = subprocess.Popen(
    [sys.executable, "-u", "app.py", "--share", "--workspace", "/content/ai_training_workspace"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
found = threading.Event()
finished = threading.Event()
url_box = {}

def _read_output():
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            if "url" not in url_box:
                match = URL_RE.search(line)
                if match:
                    url_box["url"] = match.group(0)
                    found.set()
    finally:
        finished.set()

reader = threading.Thread(target=_read_output, daemon=True)
reader.start()
try:
    while not found.wait(timeout=0.5):
        if finished.is_set():
            break
    if "url" in url_box:
        pop_app(url_box["url"])
    reader.join()
    proc.wait()
except KeyboardInterrupt:
    print("\nหยุดโปรแกรม")
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()